# Step 2: Model Training and Selection

This notebook covers:
1. Loading the preprocessed chasing dataset.
2. Training multiple Machine Learning classification models.
3. Running GroupShuffleSplit on `match_id` to avoid data leakage.
4. Evaluating accuracy, precision, recall, and F1-score.
5. Exporting the best performing model (`Random Forest`) to the `/model` directory.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import json
import os

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

## 1. Load Preprocessed Data

In [ ]:
df = pd.read_csv("../dataset/processed/win_probability_dataset.csv")
print("Dataset Shape:", df.shape)
df.head()

## 2. Define Features, Target and Groups

In [ ]:
feature_columns = [
    "target_score",
    "current_score",
    "runs_left",
    "balls_left",
    "wickets_left",
    "current_run_rate",
    "required_run_rate"
]

X = df[feature_columns]
y = df["chasing_team_won"]
groups = df["match_id"]

## 3. Split Data dynamically (Match Group Splitting)

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print(f"Training set has {X_train.shape[0]} rows, Test set has {X_test.shape[0]} rows")

## 4. Feature Scaling

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 5. Model Training and Benchmarking

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=7),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1 Score": f1
    })

results_df = pd.DataFrame(results)
results_df

## 6. Evaluate Selected Best Model (Random Forest)

In [ ]:
best_model = models["Random Forest"]
y_pred_best = best_model.predict(X_test_scaled)

print("Classification Report:")
print(classification_report(y_test, y_pred_best))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))

## 7. Export Best Model Artifacts

In [ ]:
os.makedirs("../model", exist_ok=True)

with open("../model/model.pkl", "wb") as f:
    pickle.dump(best_model, f)

with open("../model/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open("../model/feature_columns.json", "w") as f:
    json.dump(feature_columns, f)

print("Model, Scaler and Feature list exported successfully!")